# 🧑‍🍳 Neosantara AI Cookbook: Agno Telegram Bot on E2B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/agno-telegram-e2b.ipynb)

Build an **interactive Telegram bot** with the [Agno](https://github.com/agno-agi/agno) framework — powered by the **Neosantara** model provider — and package it as a reusable [E2B](https://e2b.dev) sandbox template, all from this notebook.

This mirrors E2B's [OpenClaw Telegram example](https://www.e2b.dev/docs/agents/openclaw/openclaw-telegram). Agno's [`TelegramTools`](https://docs.agno.com/tools/toolkits/social/telegram) only **sends** messages, so the bot adds the **receive** side via the Telegram Bot API `getUpdates` long-poll loop.

### 📖 Documentation
- [Agno Telegram Toolkit](https://docs.agno.com/tools/toolkits/social/telegram)
- [Agno Neosantara Model](https://docs.agno.com/examples/models/neosantara/basic)
- [E2B Templates](https://www.e2b.dev/docs/template/quickstart)

### 🔑 Prerequisites
1. A Telegram bot token from [@BotFather](https://t.me/BotFather) (run `/newbot`).
2. A [Neosantara API key](https://app.neosantara.xyz/api-keys).
3. An [E2B API key](https://e2b.dev/dashboard).

Add these as Colab Secrets (Key icon 🔑) named `TELEGRAM_TOKEN`, `NEOSANTARA_API_KEY`, and `E2B_API_KEY`, or you'll be prompted for them below.

In [ ]:
# Install the E2B SDK (template build) and Agno with the Telegram extra
!pip install -q e2b "agno[telegram]" openai

In [ ]:
# Load keys from Colab Secrets, falling back to a prompt
import os
from getpass import getpass

def load_key(name):
    if name in os.environ and os.environ[name]:
        return os.environ[name]
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    val = getpass(f"Enter {name}: ")
    os.environ[name] = val
    return val

for key in ["E2B_API_KEY", "TELEGRAM_TOKEN", "NEOSANTARA_API_KEY"]:
    load_key(key)
print("✅ Keys loaded.")

## 1. Write the bot runtime

`bot.py` long-polls Telegram for incoming messages, runs an Agno `Agent` on a Neosantara model for each one, and replies through `TelegramTools`.

In [ ]:
%%writefile bot.py
import json
import os
import sys
import time
import urllib.parse
import urllib.request

from agno.agent import Agent
from agno.models.neosantara import Neosantara
from agno.tools.telegram import TelegramTools

TELEGRAM_TOKEN = os.environ.get("TELEGRAM_TOKEN")
MODEL_ID = os.environ.get("NEOSANTARA_MODEL", "grok-4.1-fast-non-reasoning")
INSTRUCTIONS = os.environ.get(
    "AGENT_INSTRUCTIONS",
    "You are a helpful assistant chatting over Telegram. Keep replies concise.",
)
_allowed = os.environ.get("ALLOWED_CHAT_IDS", "").strip()
ALLOWED_CHAT_IDS = {c.strip() for c in _allowed.split(",") if c.strip()}

API_BASE = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
POLL_TIMEOUT = 30


def _api(method, params):
    url = f"{API_BASE}/{method}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(url, timeout=POLL_TIMEOUT + 10) as resp:
        return json.loads(resp.read().decode("utf-8"))


def build_agent(chat_id):
    return Agent(
        name="telegram",
        model=Neosantara(id=MODEL_ID),
        tools=[TelegramTools(token=TELEGRAM_TOKEN, chat_id=chat_id)],
        instructions=INSTRUCTIONS,
        markdown=False,
    )


def handle_message(chat_id, text):
    agent = build_agent(chat_id)
    agent.print_response(
        f"A Telegram user sent: {text!r}. "
        "Answer them and send the reply to the chat using your Telegram tool."
    )


def main():
    print(f"Agno Telegram bot starting (model={MODEL_ID})...", flush=True)
    offset = None
    while True:
        try:
            params = {"timeout": POLL_TIMEOUT}
            if offset is not None:
                params["offset"] = offset
            data = _api("getUpdates", params)
            if not data.get("ok"):
                print(f"getUpdates error: {data}", file=sys.stderr, flush=True)
                time.sleep(3)
                continue
            for update in data.get("result", []):
                offset = update["update_id"] + 1
                message = update.get("message") or update.get("edited_message")
                if not message:
                    continue
                text = message.get("text")
                chat_id = str(message.get("chat", {}).get("id", ""))
                if not text or not chat_id:
                    continue
                if ALLOWED_CHAT_IDS and chat_id not in ALLOWED_CHAT_IDS:
                    continue
                print(f"<- [{chat_id}] {text}", flush=True)
                try:
                    handle_message(chat_id, text)
                except Exception as exc:
                    print(f"handle_message failed: {exc}", file=sys.stderr, flush=True)
        except KeyboardInterrupt:
            break
        except Exception as exc:
            print(f"poll loop error: {exc}", file=sys.stderr, flush=True)
            time.sleep(3)


if __name__ == "__main__":
    main()

## 2. Build the E2B template

This packages `bot.py` into a reusable template named `agno-telegram` on E2B infrastructure (no local Docker required).

In [ ]:
from e2b import Template, default_build_logger

NAME = "agno-telegram"

template = (
    Template()
    .from_ubuntu_image("22.04")
    .apt_install(["python3", "python3-pip", "curl"])
    .pip_install(["agno[telegram]", "openai"])
    .set_workdir("/app")
    .copy("bot.py", "/app/bot.py")
)

info = Template.build(
    template,
    alias=NAME,
    cpu_count=2,
    memory_mb=4096,
    on_build_logs=default_build_logger(),
)
print("\nBuild complete:", info)

## 3. Launch the bot in a sandbox

Create a sandbox from the template, inject your secrets, and start the long-poll bot in the background. Then open your bot in Telegram and send it a message.

In [ ]:
from e2b import Sandbox

sbx = Sandbox.create(
    NAME,
    envs={
        "TELEGRAM_TOKEN": os.environ["TELEGRAM_TOKEN"],
        "NEOSANTARA_API_KEY": os.environ["NEOSANTARA_API_KEY"],
    },
    timeout=3600,
)

# Start the long-poll bot inside the sandbox (non-blocking)
sbx.commands.run("python /app/bot.py", background=True)
print(f"✅ Bot running in sandbox {sbx.sandbox_id}. Message your bot on Telegram!")

## 4. Clean up

Kill the sandbox when you're done to stop billing for it.

In [ ]:
sbx.kill()
print("Sandbox stopped.")